# AI Detective — AI Knowledge Investigation Assistant
### RAG & Knowledge Base Preparation Notebook (Kaggle)




## 1. Install & Imports

In [1]:
!pip install -q PyPDF2 sentence-transformers faiss-cpu transformers accelerate torch

import os, re, json, glob, pickle
from datetime import datetime
from collections import Counter

import pandas as pd
import faiss
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

PDF_DIR = "/kaggle/input/datasets/raouf158/pdf-reports"
SHEET_DIR = "/kaggle/input/datasets/raouf158/crime-recored-ttext"
os.makedirs("artifacts", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.6 MB/s eta 0:00:00:00:0100:01
Device: cuda


## 2. Extract the Knowledge Base



In [2]:

# 1. read(PDF)  

def extract_text_from_pdf(pdf_path):
  
    reader = PdfReader(pdf_path)
    return [
        {
            "document": os.path.basename(pdf_path),
            "page": i + 1,
            "text": page.extract_text() or ""  # إرجاع نص فارغ في حال فشل استخراج النص
        }
        for i, page in enumerate(reader.pages)
    ]



# 2. read (Excel أو CSV)

def read_table(path):
   
    return pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)



# 3. تنضيم الداتا

def row_to_text(row):
   
    return "\n".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))



# 4.  (Text Chunking)

def chunk_text(text, chunk_size=120, overlap=20):
  
    words = text.split()
    step = max(chunk_size - overlap, 1)
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), step) if words[i:i + chunk_size]]

# 5 (text cleaning)

def clean_text(text):
    
    return re.sub(r"\n{2,}", "\n", re.sub(r"[ \t]+", " ", text)).strip()



# 6. (Knowledge Base)

knowledge_items = []  

# --- معالجة التقارير غير المنظمة (PDF) ---
pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "**", "*.pdf"), recursive=True))
for path in pdf_paths:
    # استخراج معرف الحادثة من الجزء الأول من اسم الملف
    incident_id = os.path.basename(path).split("_")[0]
    
    # قراءة الصفحات، تنظيف النص، تقطيعه، ثم إضافته إلى القائمة
    for page in extract_text_from_pdf(path):
        for chunk in chunk_text(clean_text(page["text"])):
            knowledge_items.append({
                "text": chunk, 
                "source_type": "pdf", 
                "source_name": page["document"],
                "page": page["page"], 
                "row_number": None, 
                "incident_id": incident_id,
            })

# --- معالجة السجلات المنظمة (Excel & CSV) ---
sheet_paths = sorted(
    p for p in glob.glob(os.path.join(SHEET_DIR, "**", "*"), recursive=True) if os.path.isfile(p)
)
for path in sheet_paths:
    df = read_table(path)
    
    # تحويل كل صف في الجدول إلى عنصر معرفة مستقل
    for i, row in df.iterrows():
        knowledge_items.append({
            "text": row_to_text(row), 
            "source_type": "structured", 
            "source_name": os.path.basename(path),
            "page": None, 
            "row_number": i + 1,
            # محاولة قراءة incident_id أو استخدام رقم الصف افتراضياً
            "incident_id": str(row.get("incident_id", f"row-{i + 1}")),
        })


# 7. الطباعة والتحقق من النتيجة النهائية

print(f"{len(pdf_paths)} PDFs, {len(sheet_paths)} table file(s) -> {len(knowledge_items)} knowledge items")
knowledge_items[0]

10 PDFs, 2 table file(s) -> 32 knowledge items


{'text': 'Cross Case Investigation Summary Cross-Case Investigation Summary This document summarizes several potentially related investigation leads. Lead A — INC-001 and INC-004: INC-001 contains a witness statement describing a black sedan near a warehouse. INC-004 contains CCTV footage showing a black sedan near another warehouse. The two sources do not identify the license plate or driver. The shared vehicle description is therefore a possible connection, not confirmed proof. Lead B — INC-003 and INC-006: Both incidents occurred on River Road and involved missing construction equipment. Both investigations documented tire marks. No forensic comparison has established that the tire marks came from the same vehicle. Lead C — INC-007 and INC-010: Both records involve cargo or storage areas and a blue van',
 'source_type': 'pdf',
 'source_name': 'Cross_Case_Investigation_Summary.pdf',
 'page': 1,
 'row_number': None,
 'incident_id': 'Cross'}

## 3. Embeddings + FAISS Index




In [3]:

# 1.  
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

# 2. دالة تحويل النصوص إلى المتجهات النصية (Embeddings)

def embed_chunks(texts, model_name=EMBEDDING_MODEL_NAME):
    

    model = SentenceTransformer(model_name, device=DEVICE)
    

    embeddings = model.encode([f"passage: {t}" for t in texts], convert_to_numpy=True, show_progress_bar=True)
    
    #بسب نوع المويل 
    faiss.normalize_L2(embeddings)
    return model, embeddings



# 3. Creat _FAISS

def create_faiss_index(embeddings):
    
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index



# 4.  (Vector Search)

# index بمعني الكتاب كذا
# metadata المحتوي الاساسي بتاع الكتاب
def search_index(query, model, index, metadata, top_k=5):
  

    q = model.encode([f"query: {query}"], convert_to_numpy=True)
    faiss.normalize_L2(q)
    
  
    scores, ids = index.search(q, top_k)
    
 
    return [dict(metadata[i], score=float(s)) for s, i in zip(scores[0], ids[0]) if i != -1]



# 5. تنفيذ العمليات وإنشاء الفهرس لقاعدة المعرفة


embedding_model, embeddings = embed_chunks([k["text"] for k in knowledge_items])


index = create_faiss_index(embeddings)


print("FAISS index size:", index.ntotal)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index size: 32


## 4. Semantic Search Test

In [4]:
from collections import Counter


# 1. تحليل معرفات الحوادث (Incident IDs) واختيار أعيينات للتجربة

# حساب عدد المرات التي تكرر فيها كل incident_id داخل قاعدة المعرفة
id_counts = Counter(k["incident_id"] for k in knowledge_items)

# استخراج أكثر 3 معرفات حوادث تكراراً في البيانات
sample_ids = [i for i, _ in id_counts.most_common(3)]

# اختيار معرف رئيسي (main_id) ومعرف آخر (other_id) للتجربة مع وضع قيم افتراضية احتياطية
main_id, other_id = (sample_ids + ["INC-001", "INC-002"])[:2]



# 2. اختبار دالة البحث والاسترجاع (Search & Retrieval Test)
# ==============================================================================
# البحث في الفهرس المتجهي عن معلومات تخص الحادثة الرئيسية (main_id) لاسترجاع أفضل 3 نتائج
for r in search_index(f"Tell me about {main_id}.", embedding_model, index, knowledge_items, top_k=3):
    
    # تحديد مكان النص (رقم الصفحة إذا كان PDF، أو رقم الصف إذا كان جدولاً)
    loc = f"page {r['page']}" if r["source_type"] == "pdf" else f"row {r['row_number']}"
    
    # طباعة نسبة التشابه (Score)، اسم المصدر، الموقت، معرف الحادثة، وأول 100 حرف من النص
    print(f"[{r['score']:.3f}] {r['source_name']} ({loc}) - {r['incident_id']}: {r['text'][:100]}...")

[0.843] Cross_Case_Investigation_Summary.pdf (page 1) - Cross: the same vehicle. Lead C — INC-007 and INC-010: Both records involve cargo or storage areas and a bl...
[0.832] investigation_records.xlsx (row 1) - INC-001: incident_id: INC-001
date: 2026-08-01
time: 21:30
location: Downtown
incident_type: Warehouse Theft
...
[0.832] investigation_records.csv (row 1) - INC-001: incident_id: INC-001
date: 2026-08-01
time: 21:30
location: Downtown
incident_type: Warehouse Theft
...


## 5. LLM Setup

Same calling pattern as the reference notebook: `AutoModelForCausalLM` + `AutoTokenizer`,
sampled with `model.generate()`. `generate_text()` is the one function used everywhere
below (RAG answers, query rewriting, optional summarization).


In [5]:
LLM_MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"  # swap for a smaller instruct model if needed

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_new_tokens=350, temperature=0.7, top_k=50, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    output = llm_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True,
        top_k=top_k, top_p=top_p, temperature=temperature, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate_text("In one short sentence, what does a detective do?", max_new_tokens=40))


config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

A detective gathers clues to solve mysteries.


## 6. Conversation Memory, RAG Prompt & Pipeline

- **Full buffer memory** (`conversation_history`, required) feeds the last few turns back
  into every prompt via `build_context`.
- **Query contextualization** rewrites ambiguous follow-ups ("was it mentioned elsewhere?")
  into standalone questions using the same `generate_text` call, before retrieval.
- The **RAG prompt** explicitly forbids treating a shared detail (e.g. the same vehicle) as
  proof two cases are connected — only as a possible link.
- `rag_answer(...)` chains all of it: contextualize → retrieve → prompt → generate → log turn.


In [6]:

# 1.  (Buffer Memory)

conversation_history = [] 

# 2. دالة بناء سياق المحادثة السابقة

def build_context(history, max_turns=6):
   #بيرتب العناصر بحيث تعرف مين اللي بيتكلم اليوسر ولا  الموديل
    return "\n".join(f"{t['role'].capitalize()}: {t['content']}" for t in history[-max_turns:])



# 3. دالة إعادة صياغة السؤال بناءً على السياق (Query Contextualization)

def contextualize_query(history, question):
  
    if not history:
        return question  # إذا لم تكن هناك محادثة سابقة، يُرجع السؤال كما هو
    
    # بناء الـ Prompt الذي يوجه الـ LLM لإعادة صياغة السؤال فقط
    prompt = f"""Conversation so far:
{build_context(history, 4)}

New question: "{question}"
Rewrite it as one standalone question (resolve pronouns like "it"/"that"). Reply with ONLY the rewritten question.
Rewritten question:"""
    
    # توليد السؤال المعاد صياغته بتنظيف النواتج وأخذ السطر الأول فقط
    rewritten = generate_text(prompt, max_new_tokens=60, temperature=0.3).split("\n")[0].strip()
    return rewritten or question



# 4. دالة بناء الـ Prompt النهائي لنظام الـ RAG

def build_rag_prompt(question, evidence, history_text):
  
    # تنسيق الأدلة المسترجعة كقائمة مفصلة تحتوي المصدر، الصفحة/الصف، والمعرف
    evidence_block = "\n\n".join(
        f"[{i}] ({e['source_name']}, {'page ' + str(e['page']) if e['source_type']=='pdf' else 'row ' + str(e['row_number'])}, "
        f"incident {e['incident_id']}, score {e['score']:.2f})\n{e['text']}"
        for i, e in enumerate(evidence, 1)
    ) or "(no evidence retrieved)"
    
    # تحضير الـ Prompt الإرشادي مع تزويده بالبيانات المتاحة فقط
    return f"""You are AI Detective. Answer using ONLY the evidence and conversation below.
- Never invent facts; say so if the evidence is insufficient.
- A shared detail (e.g. same vehicle) across cases is only a POSSIBLE connection, never proof.
- Mention relevant incident IDs when synthesizing multiple sources.

Conversation so far:
{history_text or "(none)"}

Retrieved evidence:
{evidence_block}

Question: {question}
Answer:"""


# ==============================================================================
# 5. الدالة الرئيسية لمنظومة الـ RAG (RAG Execution Pipeline)
# ==============================================================================
def rag_answer(question, history, index, model, metadata, top_k=5):
   
    # الخطوة 1: فهم وإعادة صياغة السؤال بناءً على التفاعلات السابقة
    contextualized = contextualize_query(history, question)
    
    # الخطوة 2: البحث المتجهي في FAISS عن طريق السؤال المعدل
    evidence = search_index(contextualized, model, index, metadata, top_k=top_k)
    
    # الخطوة 3: صياغة الـ Prompt وتوليد الإجابة النهائية من النموذج
    answer = generate_text(build_rag_prompt(question, evidence, build_context(history)))
    
    # الخطوة 4: تحديث سجل المحادثة بإضافة السؤال والإجابة الجدد
    history += [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    
    # إرجاع قاموس مخرجات كامل يحتوي الإجابة، المصادر، والأسئلة المستخدمة
    return {
        "answer": answer, 
        "original_question": question, 
        "contextualized_question": contextualized,
        "sources": [{k: e[k] for k in ("source_type", "source_name", "page", "row_number", "incident_id", "score")} for e in evidence],
    }

## 7. Follow-up Questions & Cross-Document Reasoning

In [7]:

# 1. دالة تنسيق وطباعة نواتج الجولة الواحدة (Turn Output Formatter)

def print_turn(n, r):
   
    # طباعة فاصل ورقم الجولة مع السؤال الأصلي للمستخدم
    print(f"{'='*70}\nTurn {n} - User: {r['original_question']}")
    
    # إظهار السؤال المعاد صياغته إذا تم تعديله بناءً على السياق السابق
    if r["contextualized_question"] != r["original_question"]:
        print(f"  (contextualized: {r['contextualized_question']})")
        
    # طباعة الإجابة التوليدية للمساعد
    print(f"AI Detective: {r['answer']}\nSources:")
    
    # المرور على المصادر المسترجعة وطباعة تفاصيل كل مصدر
    for s in r["sources"]:
        # تحديد موقع النص (صفحة الـ PDF أو رقم الصف في الجدول)
        loc = f"page {s['page']}" if s["source_type"] == "pdf" else f"row {s['row_number']}"
        print(f"  - [{s['source_type']}] {s['source_name']} ({loc}) {s['incident_id']} (score {s['score']:.2f})")



# 2. محاكاة محادثة متعددة الجولات لاختبار الذاكرة واسترجاع البيانات (Multi-turn Demo)

# إنشاء ذاكرة مؤقتة خاصة بالتجربة العرضية
demo_history = []

# الجولة 1: سؤال عام عن الحادثة الرئيسية
print_turn(1, rag_answer(f"Tell me about {main_id}.", demo_history, index, embedding_model, knowledge_items))

# الجولة 2: سؤال يعتمد على السياق السابق (يستهدف استخراج تفاصيل مركبة)
print_turn(2, rag_answer("What vehicle was mentioned?", demo_history, index, embedding_model, knowledge_items))

# الجولة 3: سؤال محتوي على ضمير إشارة ("it") لربط الحادثة بحوادث أخرى
print_turn(3, rag_answer("Was it mentioned in another case?", demo_history, index, embedding_model, knowledge_items))

# الجولة 4: سؤال تحليلي يختبر عدم الهلوسة وقدرة النموذج على الاستنتاج المنطقي
print_turn(4, rag_answer("Are these two cases definitely connected?", demo_history, index, embedding_model, knowledge_items))

Turn 1 - User: Tell me about INC-001.
AI Detective: INC-001 is a Warehouse Theft incident that occurred on 2026-08-01 at 21:30 in the Downtown Warehouse District. The incident involved the theft of several boxes of electronic equipment. There were no obvious signs of forced entry at the main entrance, but the scene showed signs of recent vehicle activity at a side loading area. A nearby security guard reported seeing a black sedan leaving the loading area at approximately 21:20, though the witness could not identify the license plate or the driver. The vehicle description is considered unverified witness information, and the investigation remains open.
Sources:
  - [pdf] Cross_Case_Investigation_Summary.pdf (page 1) Cross (score 0.84)
  - [structured] investigation_records.xlsx (row 1) INC-001 (score 0.83)
  - [structured] investigation_records.csv (row 1) INC-001 (score 0.83)
  - [pdf] INC-001_Incident_Report.pdf (page 1) INC-001 (score 0.83)
  - [pdf] INC-001_Witness_Statement.pdf (p

## 8. Optional: Buffer Summary Memory



In [8]:

# 1. دالة ضغط وتلخيص سجل المحادثة القديم (Memory Summarization)

def summarize_history(history, keep_recent=4):
    
    # إذا كان حجم سجل المحادثة أقل من أو يساوي عدد التفاعلات المسموح بالحفاظ عليها، يرجع السجل كاملاً
    if len(history) <= keep_recent:
        return history
    
    # تقسيم الذاكرة إلى جزء قديم يتطلب التلخيص وجزء حديث يتم الاحتفاظ به كاملاً
    older, recent = history[:-keep_recent], history[-keep_recent:]
    
    # إرسال المحادثات القديمة للنموذج لتلخيصها في 2-3 جمل مع مراعاة دقة الحقائق ومعرفات الحوادث
    summary = generate_text(
        f"Summarize in 2-3 sentences, keeping incident IDs and facts:\n"
        + "\n".join(f"{t['role']}: {t['content']}" for t in older),
        max_new_tokens=120, 
        temperature=0.3,
    )
    
    # دمج الملخص كرسالة نظام (System Prompt) موثقة متبوعة بالتفاعلات الحديثة
    return [{"role": "system", "content": f"(summary) {summary}"}] + recent

## 9. Save & Reload Artifacts

In [9]:
import faiss
import pickle
import json
import os
from datetime import datetime


# 1. حفظ أصول النظام (Artifacts) والتكوينات على القرص الصلب

# حفظ فهرس المتجهات الخاص بـ FAISS في ملف خفيف لسرعة إعادة التحميل مستقبلاً
faiss.write_index(index, "artifacts/index.faiss")

# حفظ البيانات المرجعية والوصفية (Metadata) كاملة باستخدام Pickle
pickle.dump(knowledge_items, open("artifacts/metadata.pkl", "wb"))

# حفظ إعدادات التجربة والمعلمات الفائقة (Hyperparameters) في ملف JSON
json.dump({
    "embedding_model": EMBEDDING_MODEL_NAME, 
    "llm_model": LLM_MODEL_NAME, 
    "top_k": 5,
    "chunk_size": 120, 
    "chunk_overlap": 20, 
    "index_type": "FAISS IndexFlatIP (cosine)",
    "num_knowledge_items": len(knowledge_items), 
    "created_at": datetime.utcnow().isoformat() + "Z",  # تسجيل توقيت الحفظ بصيغة ISO القياسية
}, open("artifacts/config.json", "w"), indent=2)

# طباعة قائمة الملفات التي تم حفظها داخل مجلد artifacts للتحقق
print(os.listdir("artifacts"))



# 2. اختبار إعادة التحميل لمحاكاة تشغيل الخدمة في بيئة جديدة (Reload Test)

# قراءة فهرس المتجهات المسترجع من الملف المحلي
reloaded_index = faiss.read_index("artifacts/index.faiss")

# تحميل البيانات الوصفية المرجعية المرفقة مع الفهرس
reloaded_metadata = pickle.load(open("artifacts/metadata.pkl", "rb"))

# إجلاء اختبار بحث باستخدام الفهرس والبيانات المحملة للتأكد من سلامة التشغيل
for r in search_index(f"Tell me about {main_id}.", embedding_model, reloaded_index, reloaded_metadata, top_k=3):
    print(f"[{r['score']:.3f}] {r['incident_id']}: {r['text'][:80]}...")

['metadata.pkl', 'index.faiss', 'config.json']
[0.843] Cross: the same vehicle. Lead C — INC-007 and INC-010: Both records involve cargo or st...
[0.832] INC-001: incident_id: INC-001
date: 2026-08-01
time: 21:30
location: Downtown
incident_ty...
[0.832] INC-001: incident_id: INC-001
date: 2026-08-01
time: 21:30
location: Downtown
incident_ty...


/tmp/ipykernel_58/1674440235.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",  # تسجيل توقيت الحفظ بصيغة ISO القياسية


## 10. Evaluation

For each known incident ID: does retrieval surface evidence tagged with that ID, and
does the generated answer mention it back? Computed from real runs, not hard-coded.


In [10]:
import pandas as pd


# 1. اختبار وتقييم أداء نظام الـ RAG تلقائياً (Automated Evaluation Loop)

rows = []

# التكرار على معرفات الحوادث المختارة لاختبار جودة الاسترجاع والتوليد لكل منها
for eid in sample_ids:
    # إرسال استعلام مستقلا (مكون من ذاكرة فارغة []) لكل حادثة
    r = rag_answer(f"Tell me about {eid}.", [], index, embedding_model, knowledge_items)
    
    # تسجيل نتائج التقييم الخاصة بكل استعلام
    rows.append({
        "question": r["original_question"],
        
        # التحقق مما إذا كان المصدر المسترجع يحتوي بالفعل على معرف الحادثة المطلوب (Retrieval Accuracy)
        "retrieved_correct_evidence": eid in {s["incident_id"] for s in r["sources"]},
        
        # التحقق مما إذا كانت الإجابة المولدة تذكر معرف الحادثة بوضوح (Generative Fidelity)
        "answer_mentions_id": eid.lower() in r["answer"].lower(),
    })



# 2. تحويل النتائج إلى DataFrame واستخراج مقاييس الأداء (Metrics Calculation)

# تحويل قائمة التقييمات إلى الجدول المُنظم eval_df
eval_df = pd.DataFrame(rows)

# عرض جدول النتائج التفصيلي لكل حادثة تم اختبارها
print(eval_df)

# حساب وطباعة دقة استرجاع الأدلة الصحيحة (Retrieval Accuracy Percentage)
print(f"\nRetrieval accuracy: {eval_df['retrieved_correct_evidence'].mean()*100:.1f}%")

# حساب وطباعة معدل ذكر معرف الحادثة داخل النص المُولّد (Answer Mention Rate)
print(f"Answer ID-mention rate: {eval_df['answer_mentions_id'].mean()*100:.1f}%")

                 question  retrieved_correct_evidence  answer_mentions_id
0  Tell me about INC-001.                        True                True
1  Tell me about INC-003.                        True                True
2  Tell me about INC-004.                        True                True

Retrieval accuracy: 100.0%
Answer ID-mention rate: 100.0%


## 11. Summary

| Capability | Status |
|---|---|
| PDF + structured-record ingestion (real files, no reformatting) | ✅ |
| Chunking + multilingual E5 embeddings + FAISS | ✅ |
| LLM generation via local `AutoModelForCausalLM` (`generate_text`) | ✅ |
| RAG pipeline, evidence-grounded, correlation ≠ proof | ✅ |
| Full conversation buffer memory + query contextualization | ✅ |
| Follow-up questions + cross-document reasoning | ✅ |
| Evidence/source tracking on every answer | ✅ |
| Evaluation, artifact save + reload | ✅ |

Pretrained models only (E5 embeddings, Mistral-Nemo LLM) — nothing is trained here.
`artifacts/` (`index.faiss`, `metadata.pkl`, `config.json`) is what a later
Streamlit/FastAPI backend would load; that layer is out of scope for this notebook.


In [11]:

# ==============================================================================
# 12. FINAL AI DETECTIVE TEST SUITE
# ==============================================================================
# الهدف:
# اختبار النظام فعلياً على مجموعة أسئلة تغطي:
# 1. Direct Retrieval
# 2. PDF Evidence
# 3. Cross-Document Reasoning
# 4. Hallucination Control
# 5. Conversation Memory / Follow-up Questions
#
# ملاحظة:
# الـ Expected Answer هنا للمقارنة البشرية فقط.
# النظام نفسه لا يستخدمها أثناء الإجابة.
# ==============================================================================

test_questions = [

    # --------------------------------------------------------------------------
    # A) Direct Retrieval
    # --------------------------------------------------------------------------

    {
        "id": 1,
        "category": "Direct Retrieval",
        "question": "What happened in INC-001?",
        "expected": (
            "INC-001 was a warehouse theft at Downtown on 2026-08-01. "
            "A witness reported seeing a black sedan near the warehouse. "
            "The case is Open."
        )
    },

    {
        "id": 2,
        "category": "Direct Retrieval",
        "question": "Where did INC-003 happen?",
        "expected": "INC-003 happened at River Road."
    },

    {
        "id": 3,
        "category": "Direct Retrieval",
        "question": "What type of incident was INC-007?",
        "expected": "INC-007 was a Cargo Theft."
    },

    {
        "id": 4,
        "category": "Direct Retrieval",
        "question": "What vehicle was mentioned in INC-010?",
        "expected": "A blue van."
    },

    {
        "id": 5,
        "category": "Direct Retrieval",
        "question": "What is the status of INC-004?",
        "expected": "Under Investigation."
    },

    # --------------------------------------------------------------------------
    # B) PDF Evidence
    # --------------------------------------------------------------------------

    {
        "id": 6,
        "category": "PDF Evidence",
        "question": "Did the witness in INC-001 identify the driver of the black sedan?",
        "expected": (
            "No. The witness could not identify the driver "
            "or provide a license plate."
        )
    },

    {
        "id": 7,
        "category": "PDF Evidence",
        "question": "Does the witness statement prove that the black sedan was involved in INC-001?",
        "expected": (
            "No. The statement only reports that the vehicle was seen "
            "near the warehouse and does not establish involvement."
        )
    },

    {
        "id": 8,
        "category": "PDF Evidence",
        "question": "What did the CCTV report for INC-004 show?",
        "expected": (
            "A black sedan was seen near another warehouse, "
            "but it was not shown entering the property and the plate was unreadable."
        )
    },

    {
        "id": 9,
        "category": "PDF Evidence",
        "question": "Does the CCTV evidence confirm that INC-004 and INC-001 involve the same vehicle?",
        "expected": (
            "No. It is only an investigative lead, not confirmed proof."
        )
    },

    # --------------------------------------------------------------------------
    # C) Cross-Document Retrieval
    # --------------------------------------------------------------------------

    {
        "id": 10,
        "category": "Cross-Document",
        "question": "Which incidents are associated with a black sedan?",
        "expected": (
            "INC-001, INC-004, and INC-008."
        )
    },

    {
        "id": 11,
        "category": "Cross-Document",
        "question": "Which incidents are associated with a blue van?",
        "expected": (
            "INC-007 and INC-010."
        )
    },

    {
        "id": 12,
        "category": "Cross-Document",
        "question": "Which incidents occurred on River Road and involved equipment theft?",
        "expected": (
            "INC-003 and INC-006."
        )
    },

    {
        "id": 13,
        "category": "Cross-Document",
        "question": "What connection exists between INC-001 and INC-004?",
        "expected": (
            "Both mention a black sedan, but the available evidence "
            "does not confirm that the incidents are connected."
        )
    },

    {
        "id": 14,
        "category": "Cross-Document",
        "question": "What connection exists between INC-007 and INC-010?",
        "expected": (
            "Both involve a blue van and cargo theft-related activity, "
            "but the evidence does not confirm that the same vehicle was involved."
        )
    },

    # --------------------------------------------------------------------------
    # D) Reasoning + Hallucination Control
    # --------------------------------------------------------------------------

    {
        "id": 15,
        "category": "Reasoning",
        "question": "Is the black sedan a confirmed suspect vehicle across the cases?",
        "expected": (
            "No. It is a recurring investigative lead, "
            "but the evidence does not confirm that it is the same vehicle "
            "or prove involvement."
        )
    },

    {
        "id": 16,
        "category": "Reasoning",
        "question": "Can we conclude that the person who committed INC-001 also committed INC-004?",
        "expected": (
            "No. The available evidence does not establish that connection."
        )
    },

    {
        "id": 17,
        "category": "Hallucination Control",
        "question": "What was the license plate number of the black sedan in INC-001?",
        "expected": (
            "The license plate number is not available. "
            "The witness could not provide it."
        )
    },

    {
        "id": 18,
        "category": "Hallucination Control",
        "question": "Who was driving the black sedan in INC-001?",
        "expected": (
            "The driver was not identified in the available evidence."
        )
    },

    {
        "id": 19,
        "category": "Hallucination Control",
        "question": "What is the suspect's phone number in INC-001?",
        "expected": (
            "This information is not available in the provided data."
        )
    },
]


# ==============================================================================
# RUN ALL INDEPENDENT TESTS
# ==============================================================================

test_results = []

print("\n")
print("=" * 100)
print("                 AI DETECTIVE - FINAL TEST SUITE")
print("=" * 100)

for test in test_questions:

    # مهم:
    # كل سؤال مستقل، لذلك نستخدم memory فارغة
    # حتى لا يؤثر سؤال سابق على نتيجة السؤال الحالي.
    history = []

    try:
        result = rag_answer(
            test["question"],
            history,
            index,
            embedding_model,
            knowledge_items
        )

        answer = result["answer"]

        # استخراج الـ incident IDs من المصادر المسترجعة
        retrieved_ids = sorted(
            set(
                s["incident_id"]
                for s in result["sources"]
                if s.get("incident_id")
            )
        )

        test_results.append({
            "id": test["id"],
            "category": test["category"],
            "question": test["question"],
            "expected_answer": test["expected"],
            "actual_answer": answer,
            "retrieved_incidents": ", ".join(retrieved_ids),
            "num_sources": len(result["sources"])
        })

        print("\n" + "-" * 100)
        print(f"TEST #{test['id']} | {test['category']}")
        print("-" * 100)

        print("QUESTION:")
        print(test["question"])

        print("\nEXPECTED:")
        print(test["expected"])

        print("\nAI ANSWER:")
        print(answer)

        print("\nRETRIEVED INCIDENTS:")
        print(retrieved_ids)

        print("\nSOURCES:")
        for s in result["sources"]:
            loc = (
                f"page {s['page']}"
                if s["source_type"] == "pdf"
                else f"row {s['row_number']}"
            )

            print(
                f"  - [{s['source_type']}] "
                f"{s['source_name']} "
                f"({loc}) "
                f"{s['incident_id']} "
                f"| score={s['score']:.3f}"
            )

    except Exception as e:

        print("\n" + "-" * 100)
        print(f"TEST #{test['id']} FAILED")
        print("-" * 100)
        print("Question:", test["question"])
        print("Error:", str(e))

        test_results.append({
            "id": test["id"],
            "category": test["category"],
            "question": test["question"],
            "expected_answer": test["expected"],
            "actual_answer": f"ERROR: {str(e)}",
            "retrieved_incidents": "",
            "num_sources": 0
        })


# ==============================================================================
# SAVE RESULTS IN DATAFRAME
# ==============================================================================

test_results_df = pd.DataFrame(test_results)

print("\n\n")
print("=" * 100)
print("                     TEST RESULTS TABLE")
print("=" * 100)

display(test_results_df[
    [
        "id",
        "category",
        "question",
        "actual_answer",
        "retrieved_incidents"
    ]
])


# ==============================================================================
# EXPORT RESULTS
# ==============================================================================

test_results_df.to_csv(
    "artifacts/final_test_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to:")
print("artifacts/final_test_results.csv")





                 AI DETECTIVE - FINAL TEST SUITE

----------------------------------------------------------------------------------------------------
TEST #1 | Direct Retrieval
----------------------------------------------------------------------------------------------------
QUESTION:
What happened in INC-001?

EXPECTED:
INC-001 was a warehouse theft at Downtown on 2026-08-01. A witness reported seeing a black sedan near the warehouse. The case is Open.

AI ANSWER:
In INC-001, a small electronics warehouse in the Downtown Warehouse District reported that several boxes of networking equipment and laptop accessories were missing after the warehouse was closed on 2026-08-01 at 21:30. There were no obvious signs of forced entry at the main entrance, but a side loading area showed signs of recent vehicle activity. A nearby security guard reported seeing a dark-colored sedan leaving the loading area at approximately 21:20.

RETRIEVED INCIDENTS:
['Cross', 'INC-001', 'INC-004']

SOURCES:


,id,category,question,actual_answer,retrieved_incidents
0,1,Direct Retrieval,What happened in INC-001?,"In INC-001, a small electronics warehouse in t...","Cross, INC-001, INC-004"
1,2,Direct Retrieval,Where did INC-003 happen?,River Road,"Cross, INC-001, INC-003, INC-004, INC-010"
2,3,Direct Retrieval,What type of incident was INC-007?,INC-007 was an unauthorized access incident (I...,"Cross, INC-001, INC-003, INC-004, INC-007"
3,4,Direct Retrieval,What vehicle was mentioned in INC-010?,A blue van.,"Cross, INC-001, INC-004, INC-010"
4,5,Direct Retrieval,What is the status of INC-004?,The status of INC-004 is that it is a CCTV rep...,"Cross, INC-001, INC-003, INC-004, INC-007"
5,6,PDF Evidence,Did the witness in INC-001 identify the driver...,The witness statement in INC-001 (evidence [1]...,"Cross, INC-001, INC-004, INC-008"
6,7,PDF Evidence,Does the witness statement prove that the blac...,The witness statement does not prove that the ...,"Cross, INC-001, INC-004, INC-008"
7,8,PDF Evidence,What did the CCTV report for INC-004 show?,The CCTV report for INC-004 showed a black sed...,"Cross, INC-001, INC-004, INC-008, INC-010"
8,9,PDF Evidence,Does the CCTV evidence confirm that INC-004 an...,The CCTV evidence suggests a possible investig...,"Cross, INC-004, INC-008, INC-010"
9,10,Cross-Document,Which incidents are associated with a black se...,"Based on the provided evidence, the incidents ...","INC-001, INC-004, INC-008"



Results saved to:
artifacts/final_test_results.csv


In [12]:

# ==============================================================================
# 13. MEMORY + FOLLOW-UP TEST
# ==============================================================================

memory_test_questions = [
    "Tell me about INC-001.",
    "What vehicle was mentioned?",
    "Was it mentioned in any other incident?",
    "Does that prove the incidents are connected?",
    "Why not?"
]

memory_history = []

print("\n")
print("=" * 100)
print("                AI DETECTIVE - MEMORY TEST")
print("=" * 100)

for i, question in enumerate(memory_test_questions, start=1):

    result = rag_answer(
        question,
        memory_history,
        index,
        embedding_model,
        knowledge_items
    )

    print("\n" + "-" * 100)
    print(f"MEMORY TEST - TURN {i}")
    print("-" * 100)

    print("USER:")
    print(question)

    print("\nCONTEXTUALIZED QUESTION:")
    print(result["contextualized_question"])

    print("\nAI DETECTIVE:")
    print(result["answer"])

    print("\nSOURCES:")

    for s in result["sources"]:

        loc = (
            f"page {s['page']}"
            if s["source_type"] == "pdf"
            else f"row {s['row_number']}"
        )

        print(
            f"  - [{s['source_type']}] "
            f"{s['source_name']} "
            f"({loc}) "
            f"{s['incident_id']} "
            f"| score={s['score']:.3f}"
        )

    print("\nCURRENT MEMORY:")
    print(build_context(memory_history, max_turns=6))




                AI DETECTIVE - MEMORY TEST

----------------------------------------------------------------------------------------------------
MEMORY TEST - TURN 1
----------------------------------------------------------------------------------------------------
USER:
Tell me about INC-001.

CONTEXTUALIZED QUESTION:
Tell me about INC-001.

AI DETECTIVE:
INC-001 is a warehouse theft incident that occurred on 2026-08-01 at 21:30 in the Downtown Warehouse District. The warehouse manager reported that several boxes of networking equipment and laptop accessories were missing after the warehouse was closed. There were no obvious signs of forced entry at the main entrance, but the side loading area showed signs of recent vehicle activity. A nearby security guard witnessed a dark-colored sedan leaving the loading area at approximately 21:20. The guard could not identify the license plate or the driver. The investigation is still open, and the vehicle description is considered unverified 

In [13]:

demo_history = []

# الجولة 1: سؤال عام عن الحادثة الرئيسية
print_turn(1, rag_answer(f"i love football", demo_history, index, embedding_model, knowledge_items))

# 

Turn 1 - User: i love football
AI Detective: This information is not related to the provided evidence.
Sources:
  - [pdf] INC-001_Incident_Report.pdf (page 1) INC-001 (score 0.75)
  - [structured] investigation_records.xlsx (row 8) INC-008 (score 0.73)
  - [structured] investigation_records.csv (row 8) INC-008 (score 0.73)
  - [structured] investigation_records.xlsx (row 10) INC-010 (score 0.73)
  - [structured] investigation_records.csv (row 10) INC-010 (score 0.73)
